In [ ]:
import mlflow
import numpy as np
import polars as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.metrics import r2_score
from torch.utils.data import DataLoader, TensorDataset
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader as GeoDataLoader
from torch_geometric.nn import BatchNorm, GINConv, global_add_pool

pl.Config.set_tbl_cols(-1)       # Wyświetl wszystkie kolumny
pl.Config.set_tbl_rows(20)       # Ustaw limit wierszy (lub -1 dla wszystkich)
pl.Config.set_fmt_str_lengths(100) # Nie skracaj długich ciągów (np. SMILES)

/home/computer/Repositories/ml_chembl/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


polars.config.Config

In [2]:
df = pl.read_parquet("processed_data/ChEMBL_processed.parquet")

In [3]:
print(df.columns)

['activity_id', 'molregno', 'canonical_smiles', 'mw_freebase', 'alogp', 'hba', 'hbd', 'psa', 'rtb', 'aromatic_rings', 'qed_weighted', 'standard_value', 'standard_units', 'standard_type', 'standard_relation', 'pchembl_value', 'target_chembl_id', 'target_name', 'confidence_score', 'pIC50']


In [4]:
radius = 2
n_bits = 2048
mfpgen = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=n_bits)

def get_scaffold_smiles(smiles: str) -> str | None:
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return MurckoScaffold.MurckoScaffoldSmiles(mol=mol, includeChirality=False)

def fp_from_smiles(smiles: str) -> np.ndarray | None:
    """Zwraca fingerprint Morgana albo None dla niepoprawnego SMILES."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    try:
        return mfpgen.GetFingerprintAsNumPy(mol).astype(np.float32)
    except Exception:
        return None

# Architektura zgodna z wytycznymi
class MLPBaseline(nn.Module):
    def __init__(self, input_size=2048):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 1)
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        return self.network(x)

class GNNBaseline(torch.nn.Module):
    def __init__(self, node_features):
        super().__init__()
        nn1 = torch.nn.Sequential(torch.nn.Linear(node_features, 64), torch.nn.ReLU(), torch.nn.Linear(64, 64))
        nn2 = torch.nn.Sequential(torch.nn.Linear(64, 64), torch.nn.ReLU(), torch.nn.Linear(64, 64))
        nn3 = torch.nn.Sequential(torch.nn.Linear(64, 64), torch.nn.ReLU(), torch.nn.Linear(64, 64))
        self.conv1 = GINConv(nn1)
        self.conv2 = GINConv(nn2)
        self.conv3 = GINConv(nn3)
        self.bn1 = BatchNorm(64)
        self.bn2 = BatchNorm(64)
        self.bn3 = BatchNorm(64)
        self.lin1 = torch.nn.Linear(64, 32)
        self.lin2 = torch.nn.Linear(32, 1)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        x = self.bn1(F.relu(self.conv1(x, edge_index)))
        x = self.bn2(F.relu(self.conv2(x, edge_index)))
        x = self.bn3(F.relu(self.conv3(x, edge_index)))
        x = global_add_pool(x, batch)
        x = F.relu(self.lin1(x))
        return self.lin2(x)

In [5]:
## Uczenie
import random


def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
seed_everything(42)

In [6]:
def train_one_epoch(model, loader, optimizer, criterion, device, is_gnn=False):
    model.train()
    total_loss = 0
    for batch in loader:
        if is_gnn:
            batch = batch.to(device)
            target = batch.y.view(-1, 1)
            out = model(batch)
        else:
            x, y = batch
            x, target = x.to(device), y.to(device).view(-1, 1)
            out = model(x)

        loss = criterion(out, target)
        if torch.isnan(loss):
            print("Loss exploded to NaN! Stopping...")
            return float('nan')

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate(model, loader, device, is_gnn=False):
    model.eval()
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for batch in loader:
            if is_gnn:
                batch = batch.to(device)
                out = model(batch)
                target = batch.y.view(-1, 1)
            else:
                x, y = batch
                out = model(x.to(device))
                target = y.view(-1, 1)
            all_preds.append(out.cpu().numpy())
            all_targets.append(target.cpu().numpy())
    all_preds = np.concatenate(all_preds).ravel()
    all_targets = np.concatenate(all_targets).ravel()
    if np.isnan(all_preds).any():
        print(f"BŁĄD: Predykcje modelu zawierają NaN! (Pierwsze 5: {all_preds[:5]})")
    if np.isnan(all_targets).any():
        print("BŁĄD: Etykiety w zbiorze walidacyjnym zawierają NaN!")
    return r2_score(all_targets, all_preds)

In [7]:
df_temp = df.with_columns(
    pl.col("canonical_smiles").map_elements(fp_from_smiles, return_dtype=pl.Object).alias("fp")
)
df_clean = df_temp.filter(
    (pl.col("fp").is_not_null()) & 
    (pl.col("pIC50").is_not_null()) &
    (pl.col("pIC50").is_not_nan())
)
print(f"Liczba próbek po pełnym oczyszczeniu: {len(df_clean)}")
print(f"Odrzucono {len(df) - len(df_clean)} błędnych cząsteczek.")

[18:19:17] Explicit valence for atom # 1 P, 7, is greater than permitted
[18:19:19] WARNING: not removing hydrogen atom without neighbors
[18:19:31] Can't kekulize mol.  Unkekulized atoms: 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65 66 67 68 69 70 71 72 73 74 75 76 77
[18:20:05] Explicit valence for atom # 34 P, 7, is greater than permitted
[18:20:21] WARNING: not removing hydrogen atom without neighbors


Liczba próbek po pełnym oczyszczeniu: 561058
Odrzucono 4 błędnych cząsteczek.


In [8]:
def random_split_df(df_fp: pl.DataFrame, test_size=0.1, val_size=0.1, seed=42):
    df_shuffled = df_fp.sample(fraction=1.0, seed=seed)
    n = len(df_shuffled)
    n_test = int(test_size * n)
    n_val = int(val_size * n)
    test_df = df_shuffled[:n_test]
    val_df = df_shuffled[n_test:n_test + n_val]
    train_df = df_shuffled[n_test + n_val:]
    return train_df, val_df, test_df

def scaffold_split(df_fp: pl.DataFrame, test_size=0.1, val_size=0.1, seed=42):
    df_scaff = df_fp.with_columns(
        pl.col("canonical_smiles").map_elements(get_scaffold_smiles, return_dtype=pl.Utf8).alias("scaffold")
    )
    df_scaff = df_scaff.filter(pl.col("scaffold").is_not_null())
    scaffolds = df_scaff["scaffold"].to_list()
    rng = np.random.default_rng(seed)
    unique_scaff = list(set(scaffolds))
    rng.shuffle(unique_scaff)
    n_test = max(1, int(len(unique_scaff) * test_size))
    n_val = max(1, int(len(unique_scaff) * val_size))
    test_scaff = set(unique_scaff[:n_test])
    val_scaff = set(unique_scaff[n_test:n_test + n_val])

    def assign_split(scaff):
        if scaff in test_scaff:
            return "test"
        if scaff in val_scaff:
            return "val"
        return "train"

    df_labeled = df_scaff.with_columns(
        pl.col("scaffold").map_elements(assign_split, return_dtype=pl.Utf8).alias("split")
    )
    return (
        df_labeled.filter(pl.col("split") == "train"),
        df_labeled.filter(pl.col("split") == "val"),
        df_labeled.filter(pl.col("split") == "test"),
    )

def get_split_dfs(df_fp: pl.DataFrame, split_type="random", seed=42):
    if split_type == "scaffold":
        return scaffold_split(df_fp, seed=seed)
    if split_type == "random":
        return random_split_df(df_fp, seed=seed)
    raise ValueError("split_type must be 'random' or 'scaffold'")

def build_mlp_loaders(df_fp: pl.DataFrame, split_type="random", batch_size=64, seed=42):
    train_df, val_df, test_df = get_split_dfs(df_fp, split_type=split_type, seed=seed)

    X_train = np.stack(train_df["fp"].to_list())
    y_train = train_df["pIC50"].to_numpy().astype(np.float32)
    X_val = np.stack(val_df["fp"].to_list())
    y_val = val_df["pIC50"].to_numpy().astype(np.float32)
    X_test = np.stack(test_df["fp"].to_list())
    y_test = test_df["pIC50"].to_numpy().astype(np.float32)

    train_loader = DataLoader(
        TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train)),
        batch_size=batch_size,
        shuffle=True,
    )
    val_loader = DataLoader(
        TensorDataset(torch.from_numpy(X_val), torch.from_numpy(y_val)),
        batch_size=batch_size,
    )
    test_loader = DataLoader(
        TensorDataset(torch.from_numpy(X_test), torch.from_numpy(y_test)),
        batch_size=batch_size,
    )
    return train_loader, val_loader, test_loader

In [9]:
ATOMIC_NUM_LIST = [6, 7, 8, 9, 15, 16, 17, 35, 53]
HYBRIDIZATION_TYPES = [
    Chem.rdchem.HybridizationType.SP,
    Chem.rdchem.HybridizationType.SP2,
    Chem.rdchem.HybridizationType.SP3,
    Chem.rdchem.HybridizationType.SP3D,
    Chem.rdchem.HybridizationType.SP3D2,
]
BOND_TYPES = [
    Chem.rdchem.BondType.SINGLE,
    Chem.rdchem.BondType.DOUBLE,
    Chem.rdchem.BondType.TRIPLE,
    Chem.rdchem.BondType.AROMATIC,
]

def one_hot_encode(value, categories):
    return [1 if value == cat else 0 for cat in categories]

def mol_to_graph(smiles: str, target: float):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    
    node_feats = []
    for atom in mol.GetAtoms():
        atomic_num = atom.GetAtomicNum()
        one_hot_atomic = one_hot_encode(atomic_num, ATOMIC_NUM_LIST) if atomic_num in ATOMIC_NUM_LIST else [0] * len(ATOMIC_NUM_LIST)
        hybridization = atom.GetHybridization()
        one_hot_hybrid = one_hot_encode(hybridization, HYBRIDIZATION_TYPES) if hybridization in HYBRIDIZATION_TYPES else [0] * len(HYBRIDIZATION_TYPES)
        degree = atom.GetTotalDegree() / 4.0
        formal_charge = (atom.GetFormalCharge() + 2) / 4.0
        num_hs = atom.GetTotalNumHs() / 4.0
        is_aromatic = int(atom.GetIsAromatic())
        node_feats.append(one_hot_atomic + one_hot_hybrid + [degree, formal_charge, num_hs, is_aromatic])
    
    x = torch.tensor(node_feats, dtype=torch.float)
    
    edge_index_list = []
    edge_attr_list = []
    for bond in mol.GetBonds():
        u = bond.GetBeginAtomIdx()
        v = bond.GetEndAtomIdx()
        bond_type = bond.GetBondType()
        one_hot_bond = one_hot_encode(bond_type, BOND_TYPES) if bond_type in BOND_TYPES else [0] * len(BOND_TYPES)
        edge_index_list.extend([[u, v], [v, u]])
        edge_attr_list.extend([one_hot_bond, one_hot_bond])
    
    if edge_index_list:
        edge_index = torch.tensor(edge_index_list, dtype=torch.long).t().contiguous()
        edge_attr = torch.tensor(edge_attr_list, dtype=torch.float)
    else:
        edge_index = torch.empty((2, 0), dtype=torch.long)
        edge_attr = torch.empty((0, len(BOND_TYPES)), dtype=torch.float)
    
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=torch.tensor([target], dtype=torch.float))

def build_gnn_loaders(df_fp: pl.DataFrame, split_type="random", batch_size=64, seed=42):
    train_df, val_df, test_df = get_split_dfs(df_fp, split_type=split_type, seed=seed)

    def df_to_graphs(df_part):
        graphs = []
        for row in df_part.iter_rows(named=True):
            g = mol_to_graph(row["canonical_smiles"], float(row["pIC50"]))
            if g is not None:
                graphs.append(g)
        return graphs

    train_graphs = df_to_graphs(train_df)
    val_graphs = df_to_graphs(val_df)
    test_graphs = df_to_graphs(test_df)
    train_loader = GeoDataLoader(train_graphs, batch_size=batch_size, shuffle=True)
    val_loader = GeoDataLoader(val_graphs, batch_size=batch_size)
    test_loader = GeoDataLoader(test_graphs, batch_size=batch_size)
    return train_loader, val_loader, test_loader

In [10]:
## Konfiguracja eksperymentów (używana przez komórki poniżej)
EPOCHS_DEFAULT = 75
LR_DEFAULT = 5e-4
BATCH_SIZE_DEFAULT = 64
SEED_DEFAULT = 42

print(f"Domyślne parametry: epochs={EPOCHS_DEFAULT}, lr={LR_DEFAULT}, batch={BATCH_SIZE_DEFAULT}, seed={SEED_DEFAULT}")

Domyślne parametry: epochs=15, lr=0.0001, batch=64, seed=42


In [11]:
fp_has_nan = any(np.isnan(fp).any() for fp in df_clean["fp"])
print(f"NaN w fingerprintach: {fp_has_nan}")

NaN w fingerprintach: False


In [12]:
# Narzędzia do pojedynczego treningu i kolekcji wyników
results_table = []

def train_and_score(
    model_type: str,
    split_type: str,
    epochs: int = EPOCHS_DEFAULT,
    lr: float = LR_DEFAULT,
    batch_size: int = BATCH_SIZE_DEFAULT,
    seed: int = SEED_DEFAULT,
    log_mlflow: bool = False,
    replace_existing: bool = True,
    evaluate_test: bool = False,
):
    """Trenuje model, wybiera po walidacji, a test liczy opcjonalnie na końcu."""
    seed_everything(seed)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    if model_type == "MLP":
        train_loader, val_loader, test_loader = build_mlp_loaders(
            df_clean, split_type=split_type, batch_size=batch_size, seed=seed
        )
        model = MLPBaseline().to(device)
        is_gnn_flag = False
    elif model_type == "GNN":
        train_loader, val_loader, test_loader = build_gnn_loaders(
            df_clean, split_type=split_type, batch_size=batch_size, seed=seed
        )
        sample_graph = train_loader.dataset[0]
        model = GNNBaseline(sample_graph.num_node_features).to(device)
        is_gnn_flag = True
    else:
        raise ValueError("model_type must be 'MLP' or 'GNN'")

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = torch.nn.MSELoss()

    for _ in range(epochs):
        train_one_epoch(model, train_loader, optimizer, criterion, device, is_gnn=is_gnn_flag)

    r2_val = evaluate(model, val_loader, device, is_gnn=is_gnn_flag)
    r2_test = evaluate(model, test_loader, device, is_gnn=is_gnn_flag) if evaluate_test else None

    if log_mlflow:
        with mlflow.start_run(run_name=f"{model_type}_{split_type}"):
            mlflow.log_params({
                "model_type": model_type,
                "split_type": split_type,
                "epochs": epochs,
                "lr": lr,
                "batch_size": batch_size,
                "seed": seed,
                "evaluate_test": evaluate_test,
            })
            mlflow.log_metric("r2_val", r2_val)
            if r2_test is not None:
                mlflow.log_metric("r2_test", r2_test)
            mlflow.pytorch.log_model(model, "model")

    result = {
        "model": model_type,
        "split": split_type,
        "epochs": epochs,
        "lr": lr,
        "r2_val": r2_val,
        "r2_test": r2_test,
    }

    if replace_existing:
        results_table[:] = [
            r for r in results_table
            if not (r["model"] == model_type and r["split"] == split_type)
        ]
    results_table.append(result)

    if r2_test is None:
        print(f"{model_type} | {split_type}: val R2={r2_val:.3f}")
    else:
        print(f"{model_type} | {split_type}: val R2={r2_val:.3f}, test R2={r2_test:.3f}")
    return result

In [13]:
# MLP - random split (porównanie po walidacji)
train_and_score(model_type="MLP", split_type="random", log_mlflow=False, evaluate_test=False)

MLP | random: val R2=0.598


{'model': 'MLP',
 'split': 'random',
 'epochs': 15,
 'lr': 0.0001,
 'r2_val': 0.597922146320343,
 'r2_test': None}

In [14]:
# MLP - scaffold split (porównanie po walidacji)
train_and_score(model_type="MLP", split_type="scaffold", log_mlflow=False, evaluate_test=False)

[18:23:44] WARNING: not removing hydrogen atom without neighbors
[18:25:15] WARNING: not removing hydrogen atom without neighbors


MLP | scaffold: val R2=0.493


{'model': 'MLP',
 'split': 'scaffold',
 'epochs': 15,
 'lr': 0.0001,
 'r2_val': 0.4932953715324402,
 'r2_test': None}

In [15]:
# GNN - random split (porównanie po walidacji)
train_and_score(model_type="GNN", split_type="random", log_mlflow=False, evaluate_test=False)

[18:30:08] WARNING: not removing hydrogen atom without neighbors
[18:32:50] WARNING: not removing hydrogen atom without neighbors


GNN | random: val R2=0.092


{'model': 'GNN',
 'split': 'random',
 'epochs': 15,
 'lr': 0.0001,
 'r2_val': 0.09221756458282471,
 'r2_test': None}

In [16]:
# GNN - scaffold split (porównanie po walidacji)
train_and_score(model_type="GNN", split_type="scaffold", log_mlflow=False, evaluate_test=False)

[18:44:21] WARNING: not removing hydrogen atom without neighbors
[18:45:53] WARNING: not removing hydrogen atom without neighbors
[18:46:58] WARNING: not removing hydrogen atom without neighbors
[18:47:19] WARNING: not removing hydrogen atom without neighbors


GNN | scaffold: val R2=0.089


{'model': 'GNN',
 'split': 'scaffold',
 'epochs': 15,
 'lr': 0.0001,
 'r2_val': 0.08863222599029541,
 'r2_test': None}

In [17]:
# Zestawienie wyników w tabeli
import pandas as pd

if not results_table:
    print("Brak zebranych wyników. Uruchom najpierw komórki treningowe.")
else:
    df_results = pd.DataFrame(results_table)
    # Ranking modeli robimy po walidacji; test zostawiamy na finalny pojedynczy pomiar.
    display(df_results.sort_values(["r2_val"], ascending=False).reset_index(drop=True))

,model,split,epochs,lr,r2_val,r2_test
0,MLP,random,15,0.0001,0.597922,None
1,MLP,scaffold,15,0.0001,0.493295,None
2,GNN,random,15,0.0001,0.092218,None
3,GNN,scaffold,15,0.0001,0.088632,None


In [18]:
# Opcjonalnie: finalny test uruchom raz dla najlepszego wariantu wybranego po walidacji.
# Przykład:
# final_result = train_and_score(
#     model_type="MLP",
#     split_type="random",
#     log_mlflow=True,
#     evaluate_test=True,
#     replace_existing=False,
# )


# Wnioski z porównania modeli

- Ranking wariantów wykonuj na podstawie **R² walidacyjnego**; zbiór testowy uruchamiaj tylko raz dla finalisty.

- **Scaffold split** traktuj jako główny test uogólniania chemicznego na nowe rusztowania.

- Jeśli GNN wypada słabo względem MLP, priorytetem są: bogatsze cechy atomów/wiązań, strojenie learning rate i liczby epok, oraz kontrola jakości splitów.

- Wyniki raportuj z seedem i konfiguracją eksperymentu, aby porównania były reprodukowalne.